In [1]:
import pickle
from pathlib import Path

out_dir = Path("/Users/javier/FLAMES_30dor/gausspyp_v02.1/gausspy_validation")

FILENAME_DATA = out_dir / "real_FLAMES_validation_input.pkl"
FILENAME_DATA_DECOMP = out_dir / "real_FLAMES_validation_decomposed.pkl"
FILENAME_DF_VALIDATION = out_dir / "df_validation.pkl"
FILENAME_CONFIG = out_dir / "validation_training_config.pkl"

with open(FILENAME_DATA, "rb") as f:
    gauss_input = pickle.load(f)

with open(FILENAME_DATA_DECOMP, "rb") as f:
    data_decomp = pickle.load(f)

with open(FILENAME_DF_VALIDATION, "rb") as f:
    df_validation = pickle.load(f)

with open(FILENAME_CONFIG, "rb") as f:
    training_config = pickle.load(f)

print("gauss_input:", type(gauss_input))
print("data_decomp:", type(data_decomp))
print("df_validation:", type(df_validation))
print("training_config:", training_config)

FileNotFoundError: [Errno 2] No such file or directory: '\\Users\\javier\\FLAMES_30dor\\gausspyp_v02.1\\gausspy_validation\\real_FLAMES_validation_input.pkl'

In [ ]:
import numpy as np
import pandas as pd


def make_validation_summary(
    gauss_input,
    data_decomp,
    lambda0_nm=656.28,
    rchi2_threshold=20,
    many_components_threshold=5,
):
    """
    Build a compact validation summary table for GaussPy decompositions.

    One row corresponds to one analyzed spectrum.

    Parameters
    ----------
    gauss_input : dict
        Input dictionary used for GaussPy decomposition. Must contain:
        'data_list', 'x_values', 'errors', and 'metadata'.

    data_decomp : dict
        Output dictionary returned by GaussPy batch_decomposition.

    lambda0_nm : float, default=656.28
        Rest wavelength used to convert fitted centers into velocity.
        For Halpha, lambda0_nm ≈ 656.28 nm.

    rchi2_threshold : float, default=20
        Threshold above which the fit is flagged as large chi-square.

    many_components_threshold : int, default=5
        Threshold above which the fit is flagged as having many components.

    Returns
    -------
    summary : pandas.DataFrame
        Validation summary table.
    """

    c_kms = 299792.458

    rows = []

    n_spectra = len(gauss_input["data_list"])

    for i in range(n_spectra):

        metadata = gauss_input["metadata"][i]

        wave = np.asarray(metadata["wave"], dtype=float)

        if len(wave) > 1:
            dw = np.nanmedian(np.diff(wave))
        else:
            dw = np.nan

        wave0 = wave[0] if len(wave) > 0 else np.nan
        wave_min = np.nanmin(wave) if len(wave) > 0 else np.nan
        wave_max = np.nanmax(wave) if len(wave) > 0 else np.nan

        means_fit_chan = np.asarray(data_decomp["means_fit"][i], dtype=float)
        fwhms_fit_chan = np.asarray(data_decomp["fwhms_fit"][i], dtype=float)
        amps_fit = np.asarray(data_decomp["amplitudes_fit"][i], dtype=float)

        means_initial = np.asarray(data_decomp["means_initial"][i], dtype=float)
        fwhms_initial = np.asarray(data_decomp["fwhms_initial"][i], dtype=float)
        amps_initial = np.asarray(data_decomp["amplitudes_initial"][i], dtype=float)

        means_fit_err_chan = np.asarray(data_decomp["means_fit_err"][i], dtype=float)
        fwhms_fit_err_chan = np.asarray(data_decomp["fwhms_fit_err"][i], dtype=float)
        amps_fit_err = np.asarray(data_decomp["amplitudes_fit_err"][i], dtype=float)

        ncomp_initial = len(means_initial)
        ncomp_fit = len(means_fit_chan)

        rchi2 = to_scalar(data_decomp["best_fit_rchi2"][i])

        # Convert channel quantities to wavelength quantities.
        means_fit_nm = wave0 + means_fit_chan * dw
        fwhms_fit_nm = fwhms_fit_chan * dw

        # Convert wavelength quantities to velocity quantities.
        means_fit_kms = c_kms * (means_fit_nm - lambda0_nm) / lambda0_nm
        fwhms_fit_kms = c_kms * fwhms_fit_nm / lambda0_nm

        # Compute sigma
        sigma_obs_kms = fwhms_fit_kms / 2.335
        sigma_kms = (sigma_obs_kms**2-7.8**2-9.1**2)**0.5

        # Basic automatic flags.
        flag_zero_components = ncomp_fit == 0
        flag_many_components = ncomp_fit > many_components_threshold
        flag_large_rchi2 = np.isfinite(rchi2) and rchi2 > rchi2_threshold

        if ncomp_fit > 0:
            flag_center_outside_window = np.any(
                (means_fit_nm < wave_min) | (means_fit_nm > wave_max)
            )
            flag_nonpositive_fwhm = np.any(fwhms_fit_chan <= 0)
        else:
            flag_center_outside_window = False
            flag_nonpositive_fwhm = False

        # Simple automatic quality label.
        if flag_zero_components:
            auto_quality = "zero_components"
        elif flag_nonpositive_fwhm:
            auto_quality = "nonpositive_fwhm"
        elif flag_center_outside_window:
            auto_quality = "center_outside_window"
        elif flag_large_rchi2:
            auto_quality = "large_rchi2"
        elif flag_many_components:
            auto_quality = "many_components"
        else:
            auto_quality = "ok_candidate"

        row = {
            # Identification
            "validation_index": metadata.get("validation_index", i),
            "filename": metadata.get("filename", None),
            "object": metadata.get("object", None),
            "ra_deg": metadata.get("ra_deg", None),
            "dec_deg": metadata.get("dec_deg", None),
            "sample": metadata.get("sample", None),

            # Wavelength window
            "wave_min_nm": wave_min,
            "wave_max_nm": wave_max,
            "dw_nm": dw,

            # Component counts
            "ncomp_initial": ncomp_initial,
            "ncomp_fit": ncomp_fit,

            # Fit quality
            "best_fit_rchi2": rchi2,

            # Raw GaussPy outputs in channel units
            "means_initial_chan": means_initial,
            "fwhms_initial_chan": fwhms_initial,
            "amps_initial": amps_initial,

            "means_fit_chan": means_fit_chan,
            "fwhms_fit_chan": fwhms_fit_chan,
            "amps_fit": amps_fit,

            "means_fit_err_chan": means_fit_err_chan,
            "fwhms_fit_err_chan": fwhms_fit_err_chan,
            "amps_fit_err": amps_fit_err,

            # Converted quantities
            "means_fit_nm": means_fit_nm,
            "fwhms_fit_nm": fwhms_fit_nm,
            "means_fit_kms": means_fit_kms,
            "fwhms_fit_kms": fwhms_fit_kms,

            # Dispersion
            "sigma_obs_kms": sigma_obs_kms,
            "sigma_kms": sigma_kms,


            # Automatic flags
            "flag_zero_components": flag_zero_components,
            "flag_many_components": flag_many_components,
            "flag_large_rchi2": flag_large_rchi2,
            "flag_center_outside_window": flag_center_outside_window,
            "flag_nonpositive_fwhm": flag_nonpositive_fwhm,
            "auto_quality": auto_quality,

            # Manual validation fields
            "visual_quality": "",
            "notes": "",
        }

        rows.append(row)

    summary = pd.DataFrame(rows)

    return summary


In [ ]:
def to_scalar(value, default=np.nan):
    """
    Convert GaussPy scalar-like outputs to a float.

    Handles cases like:
    - 1.23
    - [1.23]
    - np.array([1.23])
    - None
    """
    if value is None:
        return default

    arr = np.asarray(value)

    if arr.size == 0:
        return default

    return float(arr.ravel()[0])

In [ ]:
df_validation_summary = make_validation_summary(
    gauss_input,
    data_decomp,
    lambda0_nm=656.28,
    rchi2_threshold=20,
    many_components_threshold=5,
)

display(df_validation_summary.head())

In [ ]:
df_validation_summary.columns

In [ ]:
from astropy.coordinates import SkyCoord
import astropy.units as u

coords = SkyCoord(
    ra=df_validation_summary["ra_deg"].values * u.deg,
    dec=df_validation_summary["dec_deg"].values * u.deg,
    frame="icrs"
)

df_validation_summary["RA_hms"] = coords.ra.to_string(
    unit=u.hour,
    sep=":",
    precision=3,
    pad=True
)

df_validation_summary["Dec_dms"] = coords.dec.to_string(
    unit=u.deg,
    sep=":",
    precision=3,
    alwayssign=True,
    pad=True
)

In [ ]:
df_validation_summary

In [ ]:

matches = df_validation_summary.loc[
    (df_validation_summary["RA_hms"] == '05:38:53.210') & (df_validation_summary["Dec_dms"] == '-69:05:19.900')
]
matches

In [ ]:
df_validation_summary.iloc[74]['fwhms_fit_kms']/2.35

In [ ]:
print(df_validation_summary["auto_quality"].value_counts())
print(df_validation_summary["ncomp_fit"].value_counts().sort_index())

In [ ]:
from pathlib import Path
import pickle

#out_dir = Path("/Users/javier/FLAMES_30dor/gausspy_validation")

FILENAME_SUMMARY = out_dir / "df_validation_summary.pkl"

with open(FILENAME_SUMMARY, "wb") as f:
    pickle.dump(df_validation_summary, f)

print("Saved:")
print(FILENAME_SUMMARY)

In [ ]:
FILENAME_SUMMARY_CSV = out_dir / "df_validation_summary.csv"

df_validation_summary.to_csv(FILENAME_SUMMARY_CSV, index=False)

print("Saved:")
print(FILENAME_SUMMARY_CSV)

In [ ]:
#df_validation_summary.loc[0, "visual_quality"] = "good"
#df_validation_summary.loc[0, "notes"] = "Two components recovered; residuals acceptable."

In [ ]:
import numpy as np

def add_velocity_columns_to_summary(
    df_summary,
    lambda0_nm=656.279,
    c_kms=299792.458,
):
    """
    Add velocity-space columns to a GaussPy validation summary table.

    Parameters
    ----------
    df_summary : pandas.DataFrame
        Summary table containing at least:
        'means_fit_nm' and 'fwhms_fit_nm'.

    lambda0_nm : float, default=656.28
        Rest wavelength of the emission line in nm.

    c_kms : float, default=299792.458
        Speed of light in km/s.

    Returns
    -------
    df_out : pandas.DataFrame
        Copy of the input dataframe with added velocity columns:
        'means_fit_kms' and 'fwhms_fit_kms'.
    """

    df_out = df_summary.copy()

    means_fit_kms = []
    fwhms_fit_kms = []

    for _, row in df_out.iterrows():

        means_nm = np.asarray(row["means_fit_nm"], dtype=float)
        fwhms_nm = np.asarray(row["fwhms_fit_nm"], dtype=float)

        v_centers = c_kms * (means_nm - lambda0_nm) / lambda0_nm
        v_fwhms = c_kms * fwhms_nm / lambda0_nm

        means_fit_kms.append(v_centers)
        fwhms_fit_kms.append(v_fwhms)

    df_out["means_fit_kms"] = means_fit_kms
    df_out["fwhms_fit_kms"] = fwhms_fit_kms
    df_out["lambda0_nm"] = lambda0_nm

    return df_out

In [ ]:
df_validation_summary = add_velocity_columns_to_summary(
    df_validation_summary,
    lambda0_nm=656.279,
)

display(df_validation_summary.head())

In [ ]:
cols_velocity = [
    "validation_index",
    "object",
    "ncomp_fit",
    "means_fit_nm",
    "fwhms_fit_nm",
    "means_fit_kms",
    "fwhms_fit_kms",
    "best_fit_rchi2",
    "auto_quality",
]

display(df_validation_summary[cols_velocity])

In [ ]:
import pandas as pd
import numpy as np

def make_component_table(df_summary):
    """
    Convert the validation summary table into a component-level table.

    One row corresponds to one fitted Gaussian component.
    """

    rows = []

    for _, row in df_summary.iterrows():

        ncomp = int(row["ncomp_fit"])

        means_nm = np.asarray(row["means_fit_nm"], dtype=float)
        fwhms_nm = np.asarray(row["fwhms_fit_nm"], dtype=float)
        amps = np.asarray(row["amps_fit"], dtype=float)

        means_kms = np.asarray(row["means_fit_kms"], dtype=float)
        fwhms_kms = np.asarray(row["fwhms_fit_kms"], dtype=float)

        for j in range(ncomp):
            rows.append({
                "validation_index": row["validation_index"],
                "component_index": j,
                "filename": row["filename"],
                "object": row["object"],
                "ra_deg": row["ra_deg"],
                "dec_deg": row["dec_deg"],
                "ra_hms": row["RA_hms"],
                "dec_dms": row["Dec_dms"],
                "ncomp_fit": ncomp,
                "best_fit_rchi2": row["best_fit_rchi2"],
                "auto_quality": row["auto_quality"],
                "visual_quality": row.get("visual_quality", ""),
                "mean_nm": means_nm[j],
                "fwhm_nm": fwhms_nm[j],
                "mean_kms": means_kms[j],
                "fwhm_kms": fwhms_kms[j],
                "amp": amps[j],
                "sigma_obs_kms": row["sigma_obs_kms"],
                "sigma_kms": row["sigma_kms"],
            })

    return pd.DataFrame(rows)

In [ ]:
df_components = make_component_table(df_validation_summary)

display(df_components.head())
print("Number of fitted components:", len(df_components))

In [ ]:
df_components

In [ ]:
display(df_components[[
    "validation_index",
    "component_index",
    "object",
    "mean_kms",
    "fwhm_kms",
    "amp",
    "best_fit_rchi2",
    "auto_quality",
]])

In [ ]:
df_components.columns

In [ ]:
import matplotlib.pyplot as plt

vel = df_components["mean_kms"]

vel_filtered = vel[np.abs(vel) < 1000]

plt.figure(figsize=(7, 4))
plt.hist(vel_filtered, bins=100)
plt.xlabel("Component center velocity [km/s]")
plt.ylabel("Number of components")
plt.title("Velocity centers of fitted Gaussian components | |v| < 1000 km/s")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
fwhm = df_components["fwhm_kms"]

fwhm_filtered = fwhm[(fwhm > 0) & (fwhm < 150)]

plt.figure(figsize=(7, 4))
plt.hist(fwhm_filtered, bins=100)
plt.xlabel("Component FWHM [km/s]")
plt.ylabel("Number of components")
plt.title("Velocity widths of fitted Gaussian components | 0 < FWHM < 150 km/s")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
with sns.axes_style(
    "darkgrid",
    rc={
        "axes.facecolor": "0.75",
        "figure.facecolor": "0.75",
        "grid.color": "0.55",
    }
):
    fig, ax = plt.subplots(figsize=(6,5), constrained_layout=True)
    rad_vel = ax.scatter(df_components['ra_deg'], df_components['dec_deg'],
                             s=20.0, 
                             c=df_components['mean_kms'],#-df_components['V_mean'], 
                             cmap='RdBu_r',
                             vmin=100,
                             vmax=500,
                         alpha = 0.5
                        )
    ax.invert_xaxis()
    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    #ax.set_title(r'H$\alpha$ - H$\beta$')
    fig.colorbar(rad_vel, ax=ax, label='km/s')
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(42)

jitter_ra = 0.000   # degrees
jitter_dec = 0.000  # degrees

ra_jittered = df_components["ra_deg"] + rng.normal(0, jitter_ra, size=len(df_components))
dec_jittered = df_components["dec_deg"] + rng.normal(0, jitter_dec, size=len(df_components))

cvals = df_components["mean_kms"] - 267.4

with sns.axes_style(
    "darkgrid",
    rc={
        "axes.facecolor": "0.75",
        "figure.facecolor": "0.75",
        "grid.color": "0.55",
    }
):
    fig, ax = plt.subplots(figsize=(6,5), constrained_layout=True)

    rad_vel = ax.scatter(
        ra_jittered,
        dec_jittered,
        s=75.0,
        c=cvals,
   #     cmap="RdBu_r",
        cmap="gist_rainbow_r",
        vmin=-35,
        vmax=+35,
        marker = 's'
    )

    ax.invert_xaxis()
    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")

    fig.colorbar(rad_vel, ax=ax, label="km/s")
    plt.show()

In [ ]:
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord

# Center the artificial WCS on 30 Dor / Tarantula
c0 = SkyCoord.from_name("Tarantula")

wcs = WCS(naxis=2)
wcs.wcs.ctype = ["RA---TAN", "DEC--TAN"]
wcs.wcs.cunit = ["deg", "deg"]

# Reference sky coordinate
wcs.wcs.crval = [c0.ra.deg, c0.dec.deg]

# Reference pixel
wcs.wcs.crpix = [500, 500]

# Pixel scale in degrees/pixel
# Negative RA scale gives standard astronomical orientation
wcs.wcs.cdelt = [-0.00005, 0.00005]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from astropy.visualization.wcsaxes import WCSAxes

rng = np.random.default_rng(42)

jitter_ra = 0.000
jitter_dec = 0.000

ra_jittered = df_components["ra_deg"].to_numpy() + rng.normal(
    0, jitter_ra, size=len(df_components)
)

dec_jittered = df_components["dec_deg"].to_numpy() + rng.normal(
    0, jitter_dec, size=len(df_components)
)

cvals = df_components["mean_kms"].to_numpy() - 267.4

with sns.axes_style(
    "darkgrid",
    rc={
        "axes.facecolor": "0.75",
        "figure.facecolor": "0.75",
        "grid.color": "0.55",
    }
):
    fig = plt.figure(figsize=(7, 6))

    # Explicit WCSAxes construction
    ax = WCSAxes(
        fig,
        [0.12, 0.12, 0.72, 0.78],   # [left, bottom, width, height]
        wcs=wcs,
    )

    fig.add_axes(ax)

    rad_vel = ax.scatter(
        ra_jittered,
        dec_jittered,
        s=15.0,
        c=cvals,
        cmap="RdBu_r",
        vmin=-45,
        vmax=+45,
        marker=".",
        transform=ax.get_transform("world"),
    )

    ax.coords[0].set_axislabel("Right Ascension")
    ax.coords[1].set_axislabel("Declination")

    ax.coords[0].set_major_formatter("hh:mm:ss")
    ax.coords[1].set_major_formatter("dd:mm:ss")

    ax.grid(color="0.55", alpha=0.6, linestyle="--")

    fig.colorbar(
        rad_vel,
        ax=ax,
        label=r"$v_{\rm comp} - 267.4$ km s$^{-1}$",
    )

    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from astropy.visualization.wcsaxes import WCSAxes

rng = np.random.default_rng(42)

jitter_ra = 0.000
jitter_dec = 0.000

ra_jittered = df_components["ra_deg"].to_numpy() + rng.normal(
    0, jitter_ra, size=len(df_components)
)

dec_jittered = df_components["dec_deg"].to_numpy() + rng.normal(
    0, jitter_dec, size=len(df_components)
)

cvals = df_components["mean_kms"].to_numpy() - 267.4

with sns.axes_style(
    "darkgrid",
    rc={
        "axes.facecolor": "0.75",
        "figure.facecolor": "0.75",
        "grid.color": "0.55",
    }
):
    fig = plt.figure(figsize=(7, 6))

    # Explicit WCSAxes construction
    ax = WCSAxes(
        fig,
        [0.12, 0.12, 0.72, 0.78],   # [left, bottom, width, height]
        wcs=wcs,
    )

    fig.add_axes(ax)

    rad_vel = ax.scatter(
        ra_jittered,
        dec_jittered,
        s=15.0,
        c=cvals,
        cmap="gist_rainbow_r",
        vmin=-35,
        vmax=+35,
        marker=".",
        transform=ax.get_transform("world"),
    )

    ax.coords[0].set_axislabel("Right Ascension")
    ax.coords[1].set_axislabel("Declination")

    ax.coords[0].set_major_formatter("hh:mm:ss")
    ax.coords[1].set_major_formatter("dd:mm:ss")

    ax.grid(color="0.55", alpha=0.6, linestyle="--")

    fig.colorbar(
        rad_vel,
        ax=ax,
        label=r"$v_{\rm comp} - 267.4$ km s$^{-1}$",
    )

    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from astropy.visualization.wcsaxes import WCSAxes

rng = np.random.default_rng(42)

jitter_ra = 0.000
jitter_dec = 0.000

ra_jittered = df_components["ra_deg"].to_numpy() + rng.normal(
    0, jitter_ra, size=len(df_components)
)

dec_jittered = df_components["dec_deg"].to_numpy() + rng.normal(
    0, jitter_dec, size=len(df_components)
)

cvals = df_components["sigma_kms"].to_numpy()

with sns.axes_style(
    "darkgrid",
    rc={
        "axes.facecolor": "0.75",
        "figure.facecolor": "0.75",
        "grid.color": "0.55",
    }
):
    fig = plt.figure(figsize=(7, 6))

    # Explicit WCSAxes construction
    ax = WCSAxes(
        fig,
        [0.12, 0.12, 0.72, 0.78],   # [left, bottom, width, height]
        wcs=wcs,
    )

    fig.add_axes(ax)

    rad_vel = ax.scatter(
        ra_jittered,
        dec_jittered,
        s=15.0,
        c=cvals,
        cmap="gist_rainbow_r",
        vmin=5,
        vmax= 45,
        marker=".",
        transform=ax.get_transform("world"),
    )

    ax.coords[0].set_axislabel("Right Ascension")
    ax.coords[1].set_axislabel("Declination")

    ax.coords[0].set_major_formatter("hh:mm:ss")
    ax.coords[1].set_major_formatter("dd:mm:ss")

    ax.grid(color="0.55", alpha=0.6, linestyle="--")

    fig.colorbar(
        rad_vel,
        ax=ax,
        label=r"$\sigma$",
    )

    plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
with sns.axes_style(
    "darkgrid",
    rc={
        "axes.facecolor": "0.75",
        "figure.facecolor": "0.75",
        "grid.color": "0.55",
    }
):
    fig, ax = plt.subplots(figsize=(6,5), constrained_layout=True)
    rad_vel = ax.scatter(df_components['ra_deg'], df_components['dec_deg'],
                             s=15.0, 
                             c=np.log10(df_components['amp']),#-df_components['V_mean'], 
                             cmap='magma',
                           #  vmin=100,
                           #  vmax=500,
                         alpha = 1.0,
                         marker = '.'
                        )
    ax.invert_xaxis()
    ax.set_xlabel("RA (deg)")
    ax.set_ylabel("Dec (deg)")
    #ax.set_title(r'H$\alpha$ - H$\beta$')
    fig.colorbar(rad_vel, ax=ax, label='A.D.U.')
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_context("talk")

# -------------------------------------------------------
# Input parameters
# -------------------------------------------------------
v_ref = 267.4  # Reference radial velocity [km/s]

marker_size = 15.0

# Coordinates
ra = df_components["ra_deg"].to_numpy(dtype=float)
dec = df_components["dec_deg"].to_numpy(dtype=float)

# Measurements
amp = df_components["amp"].to_numpy(dtype=float)
vel = df_components["mean_kms"].to_numpy(dtype=float)

# Valid data masks
mask_flux = (
    np.isfinite(ra)
    & np.isfinite(dec)
    & np.isfinite(amp)
    & (amp > 0)
)

mask_vel = (
    np.isfinite(ra)
    & np.isfinite(dec)
    & np.isfinite(vel)
)

# -------------------------------------------------------
# Figure
# -------------------------------------------------------
with sns.axes_style(
    "darkgrid",
    rc={
        "axes.facecolor": "0.75",
        "figure.facecolor": "0.75",
        "grid.color": "0.55",
    }
):

    fig, (ax1, ax2) = plt.subplots(
        1, 2,
        figsize=(14, 6),
        subplot_kw={"projection": wcs}
    )

    # -------------------------------------------------------
    # Left panel: Logarithm of amplitude
    # -------------------------------------------------------
    im1 = ax1.scatter(
        ra[mask_flux],
        dec[mask_flux],
        s=marker_size,
        c=np.log10(amp[mask_flux]),
        cmap="magma",
        marker=".",
        alpha=1.0,
        transform=ax1.get_transform("world")
    )

    cbar1 = fig.colorbar(
        im1,
        ax=ax1,
        label=r"$\log_{10}$ Amplitude [ADU]",
        shrink=0.8
    )

    ax1.set_xlabel("Right Ascension")
    ax1.set_ylabel("Declination")
    ax1.set_title(r"FLAMES H$\alpha$")

    ax1.coords[0].set_major_formatter("hh:mm:ss")
    ax1.coords[1].set_major_formatter("dd:mm:ss")

    ax1.grid(
        color="white",
        alpha=0.4,
        linestyle="--"
    )

    # R136 position
    ax1.scatter(
        c0.ra.deg,
        c0.dec.deg,
        transform=ax1.get_transform("world"),
        color="black",
        s=80,
        marker="*",
        zorder=5
    )

    ax1.annotate(
        "R136",
        xy=(c0.ra.deg, c0.dec.deg),
        xycoords=ax1.get_transform("world"),
        xytext=(8, 8),
        textcoords="offset points",
        color="black",
        fontsize=14,
        ha="left",
        va="bottom"
    )

    # -------------------------------------------------------
    # Right panel: Radial velocity
    # -------------------------------------------------------
    im2 = ax2.scatter(
        ra[mask_vel],
        dec[mask_vel],
        s=marker_size,
        c=vel[mask_vel] - v_ref,
        cmap="RdBu_r",
        vmin=-45,
        vmax=45,
        marker=".",
        alpha=1.0,
        transform=ax2.get_transform("world")
    )

    cbar2 = fig.colorbar(
        im2,
        ax=ax2,
        label=r"$v_{\rm comp} - 267.4$ [km/s]",
        shrink=0.8
    )

    ax2.set_xlabel("Right Ascension")
    ax2.set_ylabel("")
    ax2.set_title(r"FLAMES H$\alpha$")

    ax2.coords[0].set_major_formatter("hh:mm:ss")
    ax2.coords[1].set_major_formatter("dd:mm:ss")

    ax2.grid(
        color="white",
        alpha=0.4,
        linestyle="--"
    )

    # R136 position
    ax2.scatter(
        c0.ra.deg,
        c0.dec.deg,
        transform=ax2.get_transform("world"),
        color="black",
        s=80,
        marker="*",
        zorder=5
    )

    ax2.annotate(
        "R136",
        xy=(c0.ra.deg, c0.dec.deg),
        xycoords=ax2.get_transform("world"),
        xytext=(8, 8),
        textcoords="offset points",
        color="black",
        fontsize=14,
        ha="left",
        va="bottom"
    )

    # -------------------------------------------------------
    # Set spatial limits from FLAMES coordinates
    # -------------------------------------------------------
    mask_pos = np.isfinite(ra) & np.isfinite(dec)

    xpix, ypix = wcs.world_to_pixel_values(
        ra[mask_pos],
        dec[mask_pos]
    )

    valid_pix = np.isfinite(xpix) & np.isfinite(ypix)
    xpix = xpix[valid_pix]
    ypix = ypix[valid_pix]

    dx = np.ptp(xpix)
    dy = np.ptp(ypix)

    padding_x = 0.05 * dx
    padding_y = 0.05 * dy

    for ax in (ax1, ax2):
        ax.set_xlim(
            xpix.min() - padding_x,
            xpix.max() + padding_x
        )

        ax.set_ylim(
            ypix.min() - padding_y,
            ypix.max() + padding_y
        )

    # -------------------------------------------------------
    # Layout
    # -------------------------------------------------------
    plt.tight_layout()

    plt.savefig(
        FIGS_DIR / f"{name_export}.pdf",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

In [ ]:
FILENAME_COMPONENTS_CSV = out_dir / "df_components.csv"

df_components.to_csv(FILENAME_COMPONENTS_CSV, index=False)

print("Saved:")
print(FILENAME_SUMMARY_CSV)

In [ ]:
arr0 = df_components["ra_deg"]
arr1 = df_components["dec_deg"]
arr2 = df_components["mean_kms"].to_numpy() - 267.4
arr3 = np.log10(df_components['amp'])
arr4 = np.array([x[0] for x in df_components["sigma_kms"]])

In [ ]:
arr4

In [ ]:
temp_array = np.column_stack((arr0, arr1, arr2, arr3, arr4))

In [ ]:
temp_array[0]

In [ ]:
import json

class MyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


name = "FLAMES-30Dor-6563"
data_export_list = {
        "name": name,
        "data": temp_array,
    }

jsonfilename =  name + "-l.json"
with open(jsonfilename, "w") as f:
     json.dump(data_export_list, fp=f, indent=3, cls=MyEncoder)